In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score


# ── Config ───────────────────────────────────────────────────────────────────
MAX_VOCAB = 10000
MAX_LEN   = 100
EMBED_DIM = 64
BATCH     = 64
EPOCHS    = 10
LR        = 1e-3
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"


# ── Load Data ────────────────────────────────────────────────────────────────
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")


# ── Preprocessing ────────────────────────────────────────────────────────────
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()


train_df["tokens"] = train_df["customer_review"].apply(clean)
test_df["tokens"]  = test_df["customer_review"].apply(clean)


# Build vocab from train
counter = Counter(tok for tokens in train_df["tokens"] for tok in tokens)
vocab   = {"<PAD>": 0, "<UNK>": 1}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)


def encode(tokens):
    ids = [vocab.get(t, 1) for t in tokens[:MAX_LEN]]
    ids += [0] * (MAX_LEN - len(ids))
    return ids


train_df["ids"] = train_df["tokens"].apply(encode)
test_df["ids"]  = test_df["tokens"].apply(encode)


# ── Dataset ──────────────────────────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, ids, labels=None):
        self.ids    = torch.tensor(ids, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float) if labels is not None else None


    def __len__(self):
        return len(self.ids)


    def __getitem__(self, i):
        if self.labels is not None:
            return self.ids[i], self.labels[i]
        return self.ids[i]


# ── Train / Val Split ─────────────────────────────────────────────────────────
X = np.array(train_df["ids"].tolist())
y = train_df["feedback"].values


split = int(0.85 * len(X))
X_tr,  X_val = X[:split],  X[split:]
y_tr,  y_val = y[:split],  y[split:]


train_loader = DataLoader(ReviewDataset(X_tr, y_tr),   batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(ReviewDataset(X_val, y_val), batch_size=BATCH)
test_loader  = DataLoader(ReviewDataset(np.array(test_df["ids"].tolist())), batch_size=BATCH)


# ── Model ────────────────────────────────────────────────────────────────────
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )


    def forward(self, x):
        x = self.embed(x)       # (batch, seq_len, embed_dim)
        x = x.mean(dim=1)       # average all token embeddings → (batch, embed_dim)
        return self.fc(x).squeeze(1)








model = TextClassifier(len(vocab), EMBED_DIM).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


# ── Training ─────────────────────────────────────────────────────────────────
for epoch in range(EPOCHS):
    model.train()
    for ids, labels in train_loader:
        ids, labels = ids.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(ids), labels)
        loss.backward()
        optimizer.step()


    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for ids, labels in val_loader:
            logits = model(ids.to(DEVICE))
            preds += (torch.sigmoid(logits).cpu() >= 0.5).int().tolist()
            trues += labels.int().tolist()


    print(f"Epoch {epoch+1}/{EPOCHS}  |  Val F1: {f1_score(trues, preds):.4f}")


# ── Predict & Save ───────────────────────────────────────────────────────────
model.eval()
all_preds = []
with torch.no_grad():
    for ids in test_loader:
        logits = model(ids.to(DEVICE))
        all_preds += (torch.sigmoid(logits).cpu() >= 0.5).int().tolist()


pd.DataFrame({
    "customer_review": test_df["customer_review"],
    "feedback": all_preds
}).to_csv("submissions.csv", index=False)


print("Done — submissions.csv saved.")